# Auto Insurance Fraud Detection

An end-to-end machine-learning workflow for detecting fraudulent auto insurance claims.

Reusable data-processing, preprocessing, pipeline, and evaluation functions are kept in `src/`, while this notebook contains the analysis workflow and model comparison.


In [ ]:
# Google Colab setup
# Run this block only when starting from a fresh Colab session.

import os

if not os.path.exists("/content/Auto-Insurance-Fraud-ML"):
    !git clone https://github.com/seanhutagaol/Auto-Insurance-Fraud-ML.git /content/Auto-Insurance-Fraud-ML

%cd /content/Auto-Insurance-Fraud-ML

!pip install -q imbalanced-learn


In [ ]:
import sys
import os

sys.path.append(os.path.abspath("."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_processing import (
    load_and_clean_data,
    remove_outliers_zscore,
    feature_engineering,
    get_train_test_split,
)


## 1. Load and clean the dataset

In [ ]:
filepath = "data/insurance.csv"

df = load_and_clean_data(filepath)

print(f"Dataset shape after initial cleaning: {df.shape}")
display(df.head())


## 2. Outlier filtering and feature engineering

Numerical observations are filtered using a Z-score threshold of 3. The policy duration is then calculated from the policy binding date and incident date.


In [ ]:
df = remove_outliers_zscore(df, threshold=3.0)
df = feature_engineering(df)

print(f"Final modelling dataset shape: {df.shape}")
print(f"Fraud rate: {df['fraud_reported'].mean():.2%}")

display(df.head())


## 3. Stratified train-test split

In [ ]:
X_train, X_test, y_train, y_test = get_train_test_split(df)

print(f"Training features: {X_train.shape}")
print(f"Testing features:  {X_test.shape}")
print(f"Training fraud rate: {y_train.mean():.2%}")
print(f"Testing fraud rate:  {y_test.mean():.2%}")


## 4. Target distribution

In [ ]:
target_counts = df["fraud_reported"].value_counts().sort_index()

ax = target_counts.plot(kind="bar", figsize=(6, 4))
ax.set_title("Fraud Class Distribution")
ax.set_xlabel("Fraud Reported")
ax.set_ylabel("Number of Policies")
ax.set_xticklabels(["Non-Fraud", "Fraud"], rotation=0)

plt.tight_layout()
plt.show()


## 5. Build model pipelines

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from src.pipelines import build_model_pipeline

models = {
    "Logistic Regression": build_model_pipeline(
        LogisticRegression(
            max_iter=2000,
            random_state=42,
        )
    ),
    "Random Forest": build_model_pipeline(
        RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )
    ),
}


SMOTE is included inside the imbalanced-learn pipeline. This ensures that oversampling is performed during model fitting rather than being applied directly to the held-out test set.


## 6. Train and evaluate

In [ ]:
from src.evaluation import evaluate_model_performance, compare_models

results = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)

    cm, auc, specificity, sensitivity = evaluate_model_performance(
        pipeline,
        X_test,
        y_test,
        model_name=name,
    )

    results[name] = {
        "ROC AUC": auc,
        "Specificity": specificity,
        "Sensitivity": sensitivity,
    }


## 7. Model comparison

In [ ]:
comparison = compare_models(results)
display(comparison)


## 8. Interpretation

The main discrimination metric is ROC AUC. Specificity and sensitivity are also reported because fraud detection involves a trade-off between incorrectly flagging legitimate claims and failing to identify fraudulent claims.

- **Sensitivity:** proportion of fraudulent claims correctly identified.
- **Specificity:** proportion of legitimate claims correctly classified as non-fraud.
- **ROC AUC:** overall ranking/discrimination ability across classification thresholds.


## Repository structure

```text
Auto-Insurance-Fraud-ML/
├── data/
│   └── insurance.csv
├── src/
│   ├── __init__.py
│   ├── data_processing.py
│   ├── pipelines.py
│   └── evaluation.py
├── main_analysis.ipynb
├── README.md
└── LICENSE
```
